# PyAsc 定位与整体架构
在正式动手搭建环境之前，先来回答两个问题：**PyAsc 是什么？它内部是如何工作的？** 本节会把 PyAsc 的项目定位与整体架构一次性讲清楚，本节学习大纲如下。
- PyAsc 是什么、为什么需要它
- PyAsc 与 Ascend C 的共同点、差异点与接口映射
- PyAsc 的整体架构与五大核心模块

---


## 1. PyAsc 是什么
### 1.1 为什么需要 PyAsc
在昇腾 AI 处理器算子开发领域，传统开发模式采用 **Ascend C**（基于 C++ 扩展语法）。开发者需要了解比较多的语言细节，对于 Python 等其他语言的开发者不友好；同时，Ascend C 算子需要较多的交付件，才能向上提供 `torch.npu` 等 Python 侧的接口。

为给昇腾 CANN 体系提供原生的 Python 开发接口，**PyAsc** 应运而生。在保留 Ascend C 能力完备性的前提下：

- **降低开发语言门槛**：无需开发者熟悉 C++ 等高级模板特性。
- **Python 原生语法实现算子 kernel 开发**：开发者用 Python 函数 + 装饰器即可定义核函数。
- **直接支持 `torch.Tensor` 作为参数输入**：无需额外的 Torch 封装层。

### 1.2 PyAsc 的定义
**PyAsc** 是一种用于编写高效自定义算子的编程语言，**原生支持 Python 标准规范**。基于 PyAsc 编写的算子程序，通过编译器编译和运行时调度，运行在昇腾 AI 处理器上。

PyAsc 与 Ascend C 关系紧密，两者既有共同点，也有差异点。

### 1.3 PyAsc 与 Ascend C 的共同点

**支持相同的昇腾 AI 处理器**：两者均面向 **Atlas A2 训练/推理产品**（Ascend 910B）和 **Atlas A3 训练/推理产品**（Ascend 910C），算子最终都运行在同一类昇腾 AI 处理器上。

**支持相同的编程模型**：两者均遵循昇腾 NPU 的 Host-Device **异构并行编程模型** 与 **SPMD（Single Program Multiple Data，单程序多数据）并行计算**：

- **异构并行**：Host 侧（CPU）负责运行时管理、任务调度与资源准备，Device 侧（NPU）负责执行核函数中的实际计算。
- **SPMD 并行**：同一份核函数代码同时在多个 AI Core 上运行，每个 Core 通过 `block_idx` 区分自己处理的数据切片，从而实现并行计算。

**支持相同的基础 API 集合**：两者提供的算子开发原语保持一致，包括但不限于：

- **数据搬运**：在 Global Memory 与 Local Memory 之间搬运数据。
- **矢量计算**：加、减、乘、除等基础矢量运算。
- **矩阵计算**：基于 Cube 单元的矩阵乘等高阶计算。
- **同步**：跨流水线的 SetFlag / WaitFlag 同步原语。

### 1.4 PyAsc 与 Ascend C 的差异点

| 维度 | Ascend C | PyAsc |
| --- | --- | --- |
| 编程语言 | C++ 扩展语法 | 原生 Python |
| 类型系统 | 静态类型，需要使用模板参数表达类型（如 `LocalTensor<float>`） | 通过类型注解显式指定（如 `asc.float32`），无需模板参数 |
| 核函数定义 | 使用 `__global__ __aicore__` 修饰核函数 | 使用 `@asc.jit` 装饰器修饰核函数 |
| 编译方式 | AOT（Ahead-of-Time）静态编译，需独立编译产物 | JIT（Just-in-Time）即时编译，支持编译缓存复用 |
| 框架集成 | 需要额外的 Torch 封装层，才能在 PyTorch 中调用 | 支持 `torch.Tensor` 直接作为参数传入 |

### 1.5 PyAsc 与 Ascend C 接口 1:1 映射
**PyAsc 编程接口与 Ascend C 类库接口一一对应，旨在提供与 Ascend C 接口相同的编程能力**，Python 前端模块向 Python 用户提供完备的芯片开发能力。

PyAsc 与 Ascend C 主要接口的对应关系如下：

| 特性 | Ascend C (.asc) | PyAsc (Python) |
| --- | --- | --- |
| 核函数定义 | `__global__ __aicore__` | `@asc.jit` 装饰器 |
| GlobalTensor | `AscendC::GlobalTensor<T>` | `asc.GlobalTensor()` |
| LocalTensor | `AscendC::LocalTensor<T>` | `asc.LocalTensor()` |
| 数据搬运 | `AscendC::DataCopy()` | `asc.data_copy()` |
| 矢量加法 | `AscendC::Add()` | `asc.add()` |
| 同步事件 | `AscendC::SetFlag()/WaitFlag()` | `asc.set_flag()/wait_flag()` |
| 核函数调用 | `add_custom<<<...>>>()` | `vadd_kernel[num_blocks, stream]()` |

接口分类与 Ascend C 保持一致：高阶 API（如 Matmul）、基础 API、核心数据结构和枚举、同步与内存管理框架类接口。


## 2. PyAsc 整体架构
PyAsc 的整体架构如下图所示。

<img src="./images/pyasc_arch.png" alt="pyasc_arch"  width="700px">

在用户的视角里，使用 PyAsc 的核心动作只有两步：

1. 用 `@asc.jit` 装饰一个 Python 函数（核函数 / Device 侧函数）。
2. 在 Host 侧用 `kernel_func[core_num, stream](*args)` 调用核函数。

在这背后，PyAsc 会经过 **Python 源码 → AST → ASC-IR → Ascend C → NPU 二进制** 的完整编译流程。下图展示了 PyAsc 五大核心模块之间的关系：

<img src="./images/pyasc_flow.png" alt="pyasc_flow"  width="700px">

PyAsc 整体划分为五个核心模块。其中，**编译和运行模块、Python 前端模块、AST 转 ASC-IR 模块** 可以统称为 **前端模块**；**ASC-IR 定义模块、Ascend C 代码生成模块** 可以统称为 **后端模块**：

- **前端模块**
  - 编译和运行模块：通过 JIT 机制拉起整个编译流程，并解析运行时配置加载 Kernel 二进制执行算子。
  - Python 前端模块：提供与 Ascend C API 一一对应的 Python 编程接口，向 Python 用户提供完备的芯片开发能力。
  - AST 转 ASC-IR 模块：将 Python AST（抽象语法树）转换为 ASC-IR（AscendC-IR，Ascend C 中间表示）。
- **后端模块**
  - ASC-IR 定义模块：基于 MLIR 定义 ASC Dialect，提供与 Ascend C API 1:1 映射的 IR 结构。
  - Ascend C 代码生成模块：将 ASC-IR 翻译为等价的 Ascend C 源代码，作为毕昇编译器的输入。


---
## 课后练习
本节聚焦 PyAsc 定位、与 Ascend C 的关系，以及整体架构等关键概念，请通过以下题目检验掌握情况。

1. （判断题）PyAsc 是替代 PyTorch 的深度学习框架。

2. （判断题）PyAsc 编程接口与 Ascend C 类库接口一一对应。

3. （单选题）下列哪一个**不是** PyAsc 的核心模块？  
    A. 编译和运行模块  
    B. Python 前端模块  
    C. AST 转 ASC-IR 模块  
    D. Tensor 序列化模块  

4. （单选题）下列关于 PyAsc 的说法**正确**的是？  
    A. PyAsc 是基于 C++ 扩展语法的算子开发语言  
    B. PyAsc 是一种用于编写昇腾 AI 处理器自定义算子、原生支持 Python 标准规范的编程语言  
    C. PyAsc 不需要编译器和运行时调度，可以直接在 NPU 上执行 Python 源码  
    D. PyAsc 仅支持在 GPU 上运行  

5. （多选题）下列哪些是 PyAsc 相对于 Ascend C 的特点？  
    A. 使用原生 Python 语法实现核函数开发  
    B. 通过类型注解显式指定类型，无需模板参数  
    C. 采用 JIT 即时编译，并支持编译缓存复用  
    D. 支持 `torch.Tensor` 直接作为参数传入，无需额外的 Torch 封装层  

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.02_answer.txt
